# Change in active wildfires across Australian states/territories during the last 5 days

## Preparing Notebook

In [2]:
# import necessary libraries
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import os
print(os.getcwd())
os.chdir("/Users/silviazemp/Desktop/Uni/FS26/Python/project")


/Users/silviazemp/Desktop/Uni/FS26/Python/project


## Accessing Wildfire Data via API

In [3]:
# 1.
# access api url

## satellite: MODIS NRT -> MODIS has a better distribution of acquisition times, leaving less gaps in the final map, and Near Real Time for analysing live data. 
### However, Modis has a worse spatial resolution with 1km instead of 375m like VIIRS, but that is a trade-off I can bear
## area: '112,-44,154,-9' = bounding box coordinates for Australia 
## day range: '5' = data of the last 5 days (FIRMS does not let you load more data with one API request)
## date: None = most recent available data, so today's data

MAP_KEY = '4899a992545cbeb46f9fd0b6a025ef17'
api_url ='https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/MODIS_NRT/112,-44,154,-9/5' 

# 2.
# read in the data from URL

df_fires = pd.read_csv(api_url)


# 3.
# have a first glimpse at the data

display(df_fires.head(5))
display(df_fires.shape)
df_fires["acq_date"].unique()

# Check if a column has NaNs
print(df_fires["latitude"].hasnans)
print(df_fires["longitude"].hasnans)
print(df_fires["frp"].hasnans)
print(df_fires["acq_time"].hasnans)
print(df_fires["acq_date"].hasnans)

,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight
0,-15.60334,128.72180,309.71,1.79,1.31,2026-05-12,28,Terra,MODIS,54,6.1NRT,295.09,11.69,D
1,-15.59560,130.37132,314.37,2.40,1.49,2026-05-12,28,Terra,MODIS,67,6.1NRT,294.17,29.07,D
2,-15.53556,130.06055,311.08,2.26,1.46,2026-05-12,28,Terra,MODIS,59,6.1NRT,293.85,19.73,D
3,-15.53472,130.05479,311.07,2.26,1.46,2026-05-12,28,Terra,MODIS,59,6.1NRT,293.83,19.68,D
4,-15.46051,130.09088,315.25,2.27,1.46,2026-05-12,28,Terra,MODIS,67,6.1NRT,295.57,26.44,D


(2566, 14)

False
False
False
False
False


## Cleaning and Rearranging Data

### Omit unnessecary columns

### Adding Datetime Column with active time 

In [4]:
# 1. 
# combine the acq_date and acq_time column to one acq_datetime column and set it to an active time format with pandas function to_datetime

## acq_date is a string in the format YYYY-MM_DD, 
## while acq_time is an integer in Greenwich Mean Time (e.g. 603 meaning 6:03), 
## so it needs to be converted to string too (with astype(str)),
## fill it up to 4 numbers with zeros, so that all times have the same length (with str.zfill(4), e.g. 603 -> 0603)
## and save it as the format '%Y-%m-%d %H%M'

df_fires['acq_datetime'] = pd.to_datetime(df_fires['acq_date'] + ' ' + df_fires['acq_time'].astype(str).str.zfill(4), format='%Y-%m-%d %H%M')
df_fires.head()

print (f'Australia GMT timezone datetime value range: {df_fires['acq_datetime'].min()} to {df_fires['acq_datetime'].max()}')

# 2.
# convert GMT into local time? but we dont have just one local time

# 3.
# # Set the timestamp column as the index ?
###hourly_data = hourly_data.set_index("timestamp")

# Notice how 'timestamp' drops down a level to become the index!
###display(hourly_data.head(3))



Australia GMT timezone datetime value range: 2026-05-12 00:28:00 to 2026-05-16 08:26:00


### Converting raw coordinates into geometries

In [5]:
# convert latitude, longitude values into point geometry and make sure CRS is in EPSG 4326, as this is required for folium Maps

gdf_fires = gpd.GeoDataFrame(
    df_fires, geometry=gpd.points_from_xy(df_fires.longitude, df_fires.latitude), crs="EPSG:4326")
print(gdf_fires.crs)
gdf_fires.sample(5)

EPSG:4326


,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight,acq_datetime,geometry
1784,-34.43749,146.03630,306.83,1.05,1.02,2026-05-15,610,Aqua,MODIS,64,6.1NRT,287.89,8.00,D,2026-05-15 06:10:00,POINT (146.0363 -34.43749)
925,-14.30955,130.26463,328.09,1.35,1.15,2026-05-13,637,Aqua,MODIS,37,6.1NRT,298.21,26.66,D,2026-05-13 06:37:00,POINT (130.26463 -14.30955)
2556,-30.89336,116.93655,320.25,2.27,1.46,2026-05-16,826,Aqua,MODIS,80,6.1NRT,288.94,59.39,D,2026-05-16 08:26:00,POINT (116.93655 -30.89336)
1884,-14.03808,130.42152,321.52,3.51,1.76,2026-05-15,614,Aqua,MODIS,76,6.1NRT,295.44,68.70,D,2026-05-15 06:14:00,POINT (130.42152 -14.03808)
1159,-26.14893,150.32043,300.79,1.00,1.00,2026-05-13,2233,Terra,MODIS,42,6.1NRT,288.93,4.36,D,2026-05-13 22:33:00,POINT (150.32043 -26.14893)


## Adding a boundary GeoPackage file of States/Territories for spatial analysis

In [7]:
# 1. Load the GeoPackage of Australian states and territories and ensure CRS are matching
## Source of the GeoPackage: Australian Bureau of Statistics 
## https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs-edition-3/jul2021-jun2026/access-and-downloads/digital-boundary-files


gdf_states = gpd.read_file(
    "data/raw/ASGS_2021_Main_Structure_GDA2020.gpkg",
    layer="STE_2021_AUST_GDA2020"
).to_crs(epsg=4326)
 # check for valid geometries as sjoin was not working

# 2. Perform the spatial join
## the strict inner option is chosen, cause fires outside any Australian territories should be dropped (the bounding box includes some parts of Indonesia or Papua New Guinea)
## within is chosen as fires are point data and are either within or outside a polygon, and we only want the ones inside
gdf_joined = gpd.sjoin(gdf_fires, gdf_states, how="inner", predicate="within")

# 3. View the joined attribute table
display(gdf_joined.head(3))

# 4. Clean Data (omit columns not needed, as the attribute table is quite long now)
gdf_cleaned = gdf_joined.drop(columns=["brightness", "scan", "track", "confidence", "version", "bright_t31", "index_right", "CHANGE_FLAG_2021", "CHANGE_LABEL_2021", "AREA_ALBERS_SQKM", "ASGS_LOCI_URI_2021"])
display(gdf_cleaned.head(3))


,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,...,geometry,index_right,STATE_CODE_2021,STATE_NAME_2021,CHANGE_FLAG_2021,CHANGE_LABEL_2021,AUS_CODE_2021,AUS_NAME_2021,AREA_ALBERS_SQKM,ASGS_LOCI_URI_2021
0,-15.60334,128.72180,309.71,1.79,1.31,2026-05-12,28,Terra,MODIS,54,...,POINT (128.7218 -15.60334),4,5,Western Australia,0,No change,AUS,Australia,2.526632e+06,http://linked.data.gov.au/dataset/asgsed3/STE/5
1,-15.59560,130.37132,314.37,2.40,1.49,2026-05-12,28,Terra,MODIS,67,...,POINT (130.37132 -15.5956),6,7,Northern Territory,0,No change,AUS,Australia,1.348134e+06,http://linked.data.gov.au/dataset/asgsed3/STE/7
2,-15.53556,130.06055,311.08,2.26,1.46,2026-05-12,28,Terra,MODIS,59,...,POINT (130.06055 -15.53556),6,7,Northern Territory,0,No change,AUS,Australia,1.348134e+06,http://linked.data.gov.au/dataset/asgsed3/STE/7


,latitude,longitude,acq_date,acq_time,satellite,instrument,frp,daynight,acq_datetime,geometry,STATE_CODE_2021,STATE_NAME_2021,AUS_CODE_2021,AUS_NAME_2021
0,-15.60334,128.72180,2026-05-12,28,Terra,MODIS,11.69,D,2026-05-12 00:28:00,POINT (128.7218 -15.60334),5,Western Australia,AUS,Australia
1,-15.59560,130.37132,2026-05-12,28,Terra,MODIS,29.07,D,2026-05-12 00:28:00,POINT (130.37132 -15.5956),7,Northern Territory,AUS,Australia
2,-15.53556,130.06055,2026-05-12,28,Terra,MODIS,19.73,D,2026-05-12 00:28:00,POINT (130.06055 -15.53556),7,Northern Territory,AUS,Australia


## Spatial Analysis: Count fires per State/Territory

In [8]:
fire_count = gdf_cleaned.groupby("STATE_NAME_2021").size()
display(fire_count)

STATE_NAME_2021
New South Wales        265
Northern Territory    1085
Queensland             135
South Australia         17
Tasmania                43
Victoria                57
Western Australia      936
dtype: int64

## Preparing the Data for a Heatmap

In [10]:
# 1.
# group geometries by time (hourly resolution)
gdf_cleaned["time_bin"]= gdf_cleaned["acq_datetime"].dt.floor("d") #ist nur nötig bei hourly distribution, sonst identisch mit datetime column

data = []
time_index = []

for time, group in gdf_cleaned.groupby("time_bin"):
    
    heat_data = group[["latitude", "longitude"]].values.tolist()
    
    data.append(heat_data)
    time_index.append(str(time))

# check if it worked
gdf_cleaned.sample(5)
display(gdf_cleaned.sort_values(by=["frp"], ascending=False).head(5))

,latitude,longitude,acq_date,acq_time,satellite,instrument,frp,daynight,acq_datetime,geometry,STATE_CODE_2021,STATE_NAME_2021,AUS_CODE_2021,AUS_NAME_2021,time_bin
2544,-11.47734,130.54652,2026-05-16,654,Aqua,MODIS,1602.30,D,2026-05-16 06:54:00,POINT (130.54652 -11.47734),7,Northern Territory,AUS,Australia,2026-05-16
2545,-11.47594,130.55582,2026-05-16,654,Aqua,MODIS,1557.64,D,2026-05-16 06:54:00,POINT (130.55582 -11.47594),7,Northern Territory,AUS,Australia,2026-05-16
701,-17.49049,124.96676,2026-05-13,635,Aqua,MODIS,1374.10,D,2026-05-13 06:35:00,POINT (124.96676 -17.49049),5,Western Australia,AUS,Australia,2026-05-13
698,-17.49764,124.96021,2026-05-13,635,Aqua,MODIS,985.05,D,2026-05-13 06:35:00,POINT (124.96021 -17.49764),5,Western Australia,AUS,Australia,2026-05-13
2542,-11.48627,130.54796,2026-05-16,654,Aqua,MODIS,913.65,D,2026-05-16 06:54:00,POINT (130.54796 -11.48627),7,Northern Territory,AUS,Australia,2026-05-16


## Visualising the data with a folium map with State/Territory Polygons and an animated Heatmap of the Fire Distribution

In [ ]:
import folium
from folium.plugins import HeatMapWithTime
from folium.plugins import MarkerCluster
import numpy as np

# 1. Create basemap for the extent of Australia
aus_map = folium.Map(
    location=[-25.5649, 133.1234], # use the coordinates of Australia's centre (25°56′49.3″S, 133°12′34.7″E) for the location
    zoom_start=5,
    tiles=None  
)
# custom the name of the Basemap that will be shown in the Layer Control
folium.TileLayer(
    tiles="CartoDB DarkMatter", # a dark basemap to nicely contrast the heatmap and marker clusters
    name="Basemap"
).add_to(aus_map)

# 2. Add State/Territory Polygons
folium.GeoJson(
    gdf_states,
    name="States/Territories",
    tooltip=folium.GeoJsonTooltip(
        fields=["STATE_NAME_2021"],
        aliases=["State/Territory:"]
    ),
    style_function=lambda feature:{
        "fillColor": "darkgray", 
        "color": "white",
        "weight": 0.5,
    }
).add_to(aus_map)

# 3. Create heatmap with Fire Data that shows intensity of different fires (frp)
HeatMapWithTime(
    data,
    name="Heatmap",
    index=time_index,
    radius=8,
    auto_play=True,
    max_opacity=0.8,
    gradient={  
        0.2: "blue",
        0.4: "lime",
        0.6: "yellow",
        0.8: "orange",
        1.0: "red"
    }
).add_to(aus_map)

# 4. Create Marker Cluster Layer with number of fires

# create an empty group and ad it to the map
marker_cluster = MarkerCluster(name="Wildfire Clusters").add_to(aus_map)
# iterate through the GeoDataFrame
for idx, row in gdf_cleaned.iterrows():
    lat = row.geometry.y
    lon = row.geometry.x

    tooltip_text = f"Fire Radiative Power: {row['frp']} Megawatts"

    folium.Marker(
        location=[lat, lon],
        icon=folium.Icon(color="orange", icon="fire", prefix="fa"),
        tooltip=tooltip_text
    ).add_to(marker_cluster)

folium.LayerControl().add_to(aus_map)


# save the map (display does not work bc data file is too big)
aus_map.save("animated_heatmap_with_clusters.html")